In [ ]:
import os
os.environ["POLYGON_API_KEY"] = "xyz"  

### Step 1: Set up Imports & Configuration

In [ ]:
# Imports & Configuration
import os, math, numpy as np, pandas as pd
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
import plotly.graph_objects as go
from statsmodels.regression.linear_model import OLS
from statsmodels.tools.tools import add_constant
from polygon import RESTClient
import warnings
import polars as pl
warnings.filterwarnings("ignore")

# User Configuration
#TECH_TICKERS   = ["MSFT","AMZN","GOOGL","META","AAPL","NVDA","ORCL","ADBE","CRM","NOW"]
#RETAIL_TICKERS = ["TGT","WMT","COST","HD","LOW","TJX","ROST","BBY","KR","AMZN"]
# UNIVERSE DEFINITION
RETAIL_TICKERS = [
    "WMT",   # Global mass retail
    "COST",  # Wholesale retail
    "HD",    # Home improvement retail
    "TJX",   # Off-price retail
    "LOW",   # Home improvement retail
    "CVS",   # Consumer health retail
    "ROST",  # Discount apparel retail
    "AZO",   # Automotive retail
    "TGT",   # Mass retail
    "KR",    # Grocery retail
    "DG",    # Discount retail
    "TSCO",  # Rural & lifestyle retail
    "DLTR",  # Discount retail
    "ULTA",  # Specialty beauty retail
    "LULU",  # Specialty apparel retail
    "WSM",   # Home goods retail
    "CDW",   # Enterprise / tech retail
    "BURL",  # Off-price apparel retail
    "DECK",  # Branded footwear & apparel retail
    "BBY",   # Consumer electronics retail
    "W",     # Online home retail
    "BJ",    # Warehouse retail
    "DDS",   # Department store retail
    "GAP",   # Apparel retail
    "GME",   # Specialty entertainment retail
    "FIVE",  # Discount specialty retail
    "ACI",   # Grocery retail
    "URBN",  # Apparel retail
    "FCFS",  # Specialty financial retail
    "LAD",   # Automotive retail
    "MUSA",  # Fuel & convenience retail
    "SFM",   # Grocery retail
    "OLLI",  # Discount closeout retail
    "PAG",   # Automotive retail
    "SIG",   # Jewelry retail
    "ASO",   # Sporting goods retail
    "RH",    # Luxury home retail
    "M",     # Department store retail
    "ANF",   # Apparel retail
    "BBWI",  # Specialty personal care retail
    "PSMT",  # Membership grocery retail
    "OPEN",  # Automotive retail platform
    "KSS",   # Department store retail
    "AAP",   # Auto parts retail
    "GO",    # Discount grocery retail
    "EYE",   # Optical retail
    "ARHS"   # Home furnishings retail
]

TECH_TICKERS = [
    "NVDA",  # AI & GPU computing
    "AAPL",  # Consumer technology ecosystem
    "GOOG",  # Internet & AI platforms
    "MSFT",  # Cloud & enterprise software
    "AMZN",  # Cloud + platform technology
    "META",  # Digital platforms & AI
    "AVGO",  # Semiconductor infrastructure
    "TSLA",  # Technology-driven manufacturing
    "ORCL",  # Enterprise software & databases
    "PLTR",  # Data analytics & AI software
    "NFLX",  # Streaming technology platform
    "AMD",   # Semiconductor design
    "CSCO",  # Networking hardware & software
    "IBM",   # Enterprise IT & hybrid cloud
    "MU",    # Memory semiconductors
    "CRM",   # Enterprise SaaS
    "APP",   # Mobile advertising technology
    "AMAT",  # Semiconductor equipment
    "LRCX",  # Semiconductor equipment
    "QCOM",  # Wireless semiconductor technology
    "INTU",  # Financial software platforms
    "INTC",  # Semiconductor manufacturing
    "UBER",  # Platform technology & logistics
    "NOW",   # Enterprise workflow software
    "TXN",   # Analog semiconductors
    "KLAC",  # Semiconductor process control
    "ANET",  # Cloud networking
    "ADBE",  # Creative & marketing software
    "ADI",   # Analog semiconductors
    "PANW",  # Cybersecurity software
    "CRWD",  # Cybersecurity platform
    "ADP",   # Enterprise data & payroll systems
    "DASH",  # Platform logistics technology
    "DELL",  # Enterprise hardware & infrastructure
    "SNPS",  # Semiconductor design software
    "CDNS",  # Electronic design automation
    "SNOW",  # Cloud data platforms
    "EQIX",  # Data center infrastructure
    "MRVL",  # Semiconductor connectivity
    "NET",   # Cloud security & networking
    "COIN",  # Financial technology platform
    "FTNT",  # Cybersecurity appliances & software
    "ADSK",  # Design & engineering software
    "PYPL",  # Payments technology
    "WDAY",  # Enterprise HR & finance software
    "DDOG",  # Cloud monitoring & observability
    "ROP",   # Software-intensive industrial tech
    "MSTR"   # Enterprise analytics / crypto proxy
]
WINDOWS        = [5, 10, 15, 20]
Z_ENTRY        = 1.0
Z_EXIT         = 0.5
ZSCORE_METHODS = ["standard", "ewma", "adaptive"]
POSITION_SCHEME = "fixed_50_50"
VOL_WINDOW = 20

In [3]:
# Cost Model
@dataclass
class CostModel:
    commission_bps: float = 1.0
    slippage_bps: float = 3.0
    borrow_bps_per_day: float = 5.0
    fixed_fee_per_trade: float = 0.0

COSTS = CostModel()

### Step 2: Load the data

In [4]:
# Dates
TODAY = datetime.now(timezone.utc).date()
START = TODAY - timedelta(days=365*2 + 5)
END   = TODAY

# Polygon Key
POLYGON_API_KEY = os.getenv("POLYGON_API_KEY", "")
if not POLYGON_API_KEY:
    raise RuntimeError("Please set POLYGON_API_KEY")
print("Config loaded — Tech × Retail grid, Polygon data, dark theme.")

# === Data Loader ===
def _client(): return RESTClient(POLYGON_API_KEY)

def fetch_polygon_agg(ticker, start, end):
    c = _client()
    rows = []
    for bar in c.list_aggs(
        ticker=ticker, multiplier=1, timespan="day",
        from_=start.strftime("%Y-%m-%d"),
        to=end.strftime("%Y-%m-%d"),
        adjusted=True, limit=50000
    ):
        rows.append({
            "ts": pd.to_datetime(bar.timestamp, unit="ms", utc=True).tz_convert(None).date(),
            "close": bar.close
        })
    df = pd.DataFrame(rows).drop_duplicates("ts").set_index("ts").sort_index()
    df.index.name = "Date"
    df.columns = [ticker]
    return df

price_cache = {}
all_tickers = list(set(TECH_TICKERS + RETAIL_TICKERS))
for ticker in all_tickers:
    try:
        price_cache[ticker] = fetch_polygon_agg(ticker, pd.Timestamp(START), pd.Timestamp(END))
    except Exception as e:
        print(f"Failed to fetch {ticker}: {e}")

def load_prices_cached(primary, pair):
    p1 = price_cache.get(primary)
    p2 = price_cache.get(pair)
    if p1 is None or p2 is None:
        raise ValueError(f"Missing data for {primary} or {pair}")
    data = p1.join(p2, how="inner").dropna()
    returns = data.pct_change().dropna()
    return data, returns

Config loaded — Tech × Retail grid, Polygon data, dark theme.
Failed to fetch PANW: HTTPSConnectionPool(host='api.polygon.io', port=443): Max retries exceeded with url: /v2/aggs/ticker/PANW/range/1/day/2023-12-30/2026-01-03?adjusted=true&limit=50000 (Caused by ResponseError('too many 429 error responses'))
Failed to fetch NOW: HTTPSConnectionPool(host='api.polygon.io', port=443): Max retries exceeded with url: /v2/aggs/ticker/NOW/range/1/day/2023-12-30/2026-01-03?adjusted=true&limit=50000 (Caused by ResponseError('too many 429 error responses'))
Failed to fetch TSCO: HTTPSConnectionPool(host='api.polygon.io', port=443): Max retries exceeded with url: /v2/aggs/ticker/TSCO/range/1/day/2023-12-30/2026-01-03?adjusted=true&limit=50000 (Caused by ResponseError('too many 429 error responses'))
Failed to fetch TSLA: HTTPSConnectionPool(host='api.polygon.io', port=443): Max retries exceeded with url: /v2/aggs/ticker/TSLA/range/1/day/2023-12-30/2026-01-03?adjusted=true&limit=50000 (Caused by Res

### Step 3: Set up Metrics

In [5]:
# Metrics
def annualize_mean(r): return r.mean() * 252
def annualize_vol(r): return r.std() * math.sqrt(252)
#def sharpe_ratio(r): return r.mean()/r.std()*math.sqrt(252) if r.std()>0 else np.nan
def sharpe_ratio(r, risk_free_rate_annual=0.05):
    rf_daily = risk_free_rate_annual / 252
    excess = r - rf_daily
    return excess.mean() / excess.std() * math.sqrt(252) if r.std() > 0 else np.nan
def sortino_ratio(r):
    d = np.minimum(r,0)
    ds = d.std()
    return r.mean()/ds*math.sqrt(252) if ds>0 else np.nan
def max_dd(r):
    cum=(1+r).cumprod(); peak=cum.cummax(); return (cum/peak-1).min()
def perf_summary(lbl, r):
    return {
        "Label": lbl,
        "AnnRet": annualize_mean(r),
        "AnnVol": annualize_vol(r),
        "Sharpe": sharpe_ratio(r),
        "Sortino": sortino_ratio(r),
        "MaxDD": max_dd(r),
        "TotalRet": (1+r).prod()-1
    }

### Step 4: Z-score Strategy

In [6]:
class ZScoreEngine:
    """
    Corrected version:
    - half_life() is now internal and walk-forward
    - adaptive z-score is computed with rolling half-life, using only past data
    - no future information is used at any point
    """

    def __init__(self, series):
        self.series = series.dropna()

    # --- AR(1) half-life estimator for a given window of data ---
    def _estimate_half_life(self, s):
        if len(s) < 3:
            return np.nan
        try:
            lag = s.shift(1).dropna()
            y = s.iloc[1:]
            X = add_constant(lag.values)
            beta = OLS(y, X).fit().params[1]
            if 0 < beta < 1:
                return -np.log(2) / np.log(beta)
            else:
                return np.nan
        except Exception:
            return np.nan

    # --- Walk-forward half-life time series ---
    def _rolling_half_life(self, window):
        s = self.series
        hl = pd.Series(index=s.index, dtype=float)

        # For each t >= window, compute HL from s[t-window:t]
        for i in range(window, len(s)):
            past_window = s.iloc[i - window:i]   # only past data
            hl.iloc[i] = self._estimate_half_life(past_window)

        return hl

    # --- Master z-score function ---
    def zscore(self, method="adaptive", window=60, vol_window=20):
        s = self.series

        # -----------------------------
        # Standard Rolling Z-score
        # -----------------------------
        if method == "standard":
            mu = s.rolling(window).mean()
            sd = s.rolling(window).std()
            return (s - mu) / sd

        # -----------------------------
        # EWMA Z-score
        # -----------------------------
        elif method == "ewma":
            mu = s.ewm(span=window, adjust=False).mean()
            sd = s.ewm(span=window, adjust=False).std()
            return (s - mu) / sd

        # -----------------------------
        # ADAPTIVE Z-SCORE (FIXED)
        # -----------------------------
        elif method == "adaptive":

            # 1) Walk-forward half-life estimate
            hl_series = self._rolling_half_life(window)

            # 2) Convert HL into a span (fallback = window)
            span_series = hl_series.apply(
                lambda h: int(np.clip(2.5 * h, 20, 120)) 
                if pd.notna(h) and h > 0 
                else window
            )

            # Walk-forward EWMA mean & std using span_t
            mu = pd.Series(index=s.index, dtype=float)
            sd = pd.Series(index=s.index, dtype=float)
            z  = pd.Series(index=s.index, dtype=float)

            for i in range(len(s)):
                # Need at least `window` points before estimating anything
                if i < window:
                    mu.iloc[i] = np.nan
                    sd.iloc[i] = np.nan
                    z.iloc[i]  = np.nan
                    continue

                span_i = span_series.iloc[i]

                # EWMA over the past data up to time t (walk-forward)
                ewm_obj = s.iloc[:i+1].ewm(span=span_i, adjust=False)

                mu_i = ewm_obj.mean().iloc[-1]
                sd_i = ewm_obj.std().iloc[-1]

                mu.iloc[i] = mu_i
                sd.iloc[i] = sd_i
                if sd_i > 0:
                    z.iloc[i] = (s.iloc[i] - mu_i) / sd_i
                else:
                    z.iloc[i] = np.nan

            # 3) Z-vol normalization
            z_vol = z.rolling(vol_window).std()
            z_adj = z / z_vol
            return z_adj

        else:
            raise ValueError(f"Unknown method: {method}")


### Step 5: Set up the Signals and Costs

In [7]:
# Signals and Costs
def trading_signals(z, entry, exit):
    sig=pd.Series(0,index=z.index)
    state=0
    for i,v in enumerate(z):
        if np.isnan(v): sig.iloc[i]=state
        else:
            if state==0:
                if v<-entry: state=1
                elif v>entry: state=-1
            elif abs(v)<exit: state=0
            sig.iloc[i]=state
    return sig

def apply_costs(weights, costs: CostModel):
    turnover=weights.diff().abs().sum(axis=1)
    trade_cost=-(costs.commission_bps+costs.slippage_bps)/1e4 * turnover
    borrow_cost=-(costs.borrow_bps_per_day/1e4)*weights.clip(upper=0).abs().sum(axis=1)
    return trade_cost + borrow_cost

### Step 6: Set up Strategy Logic

In [8]:
# Strategy Core
def mean_reversion_strategy(data, returns, primary, pair, window, entry, exit, method, costs):
    s=data[primary]
    engine=ZScoreEngine(s)
    z=engine.zscore(method,window)
    sig=trading_signals(z,entry,exit)
    positions=pd.DataFrame(index=data.index)
    positions[primary]=0.5*sig; positions[pair]=-0.5*sig
    pnl=(positions.shift(1)*returns).sum(axis=1)
    pnl_net=pnl+apply_costs(positions,costs)
    return pnl_net.dropna(), z, positions

### Step 7: Cross Grid Backtest Set Up

In [ ]:
# Cross-Grid Backtest
results=[]
for tech in TECH_TICKERS:
    for retail in RETAIL_TICKERS:
        if tech == retail: continue
        try:
            data, rets = load_prices_cached(tech, retail)
            for w in WINDOWS:
                for m in ZSCORE_METHODS:
                    pnl, _, _ = mean_reversion_strategy(data, rets, tech, retail,
                                                        w, Z_ENTRY, Z_EXIT, m, COSTS)
                    s = perf_summary(f"{tech}-{retail}", pnl)
                    s.update({"Primary": tech, "Pair": retail, "Method": m, "Window": w})
                    results.append(s)
        except Exception as e:
            print(f"{tech}-{retail} failed: {e}")

cross_df = pd.DataFrame(results).sort_values(by="Sharpe", ascending=False).reset_index(drop=True)
display(
    # get rid of `[cross_df["Sharpe"] > 1]` or apply other filters
    # to manipulate the display of the dataframe
    cross_df[cross_df["Sharpe"] > 1].style.format({
        "AnnRet":"{:.2%}","AnnVol":"{:.2%}","Sharpe":"{:.2f}",
        "Sortino":"{:.2f}","MaxDD":"{:.2%}","TotalRet":"{:.2%}"
    }).background_gradient(subset=["Sharpe"], cmap="Greens")
)

,Label,AnnRet,AnnVol,Sharpe,Sortino,MaxDD,TotalRet,Primary,Pair,Method,Window
0,SQ-BBY,40.83%,19.50%,1.84,3.60,-10.23%,56.48%,SQ,BBY,standard,5
1,SNOW-BBY,47.21%,25.47%,1.66,4.14,-17.53%,141.28%,SNOW,BBY,standard,20
2,SQ-TJX,34.95%,18.17%,1.65,3.00,-13.01%,46.65%,SQ,TJX,standard,5
3,SQ-WMT,36.23%,19.16%,1.63,3.03,-11.37%,48.52%,SQ,WMT,standard,5
4,SQ-KR,37.95%,20.46%,1.61,2.95,-10.19%,51.03%,SQ,KR,standard,5
5,SQ-COST,34.06%,18.21%,1.60,2.90,-9.86%,45.15%,SQ,COST,standard,5
6,SQ-TSCO,31.41%,18.87%,1.40,2.78,-9.70%,40.60%,SQ,TSCO,standard,5
7,SQ-LULU,21.71%,12.06%,1.39,3.23,-6.92%,27.31%,SQ,LULU,ewma,5
8,PYPL-AMZN,24.81%,14.31%,1.38,3.33,-12.50%,60.91%,PYPL,AMZN,adaptive,10
9,MSFT-BBY,26.16%,15.31%,1.38,3.81,-7.92%,64.89%,MSFT,BBY,ewma,15


### Step 8: Visualize Top Pairs

In [20]:
# Visualize Top Pairs
TOP_N = 50

top_subset = cross_df.head(TOP_N)
fig = go.Figure()

all_curves = []

# One consistent line color for all individual strategies
uniform_color = "rgba(0, 255, 150, 0.12)"

for _, row in top_subset.iterrows():
    tech, retail, method, window = row["Primary"], row["Pair"], row["Method"], row["Window"]
    try:
        data, rets = load_prices_cached(tech, retail)
        pnl, _, _ = mean_reversion_strategy(
            data, rets, tech, retail, window, Z_ENTRY, Z_EXIT, method, COSTS
        )
        cum = (1 + pnl).cumprod()
        all_curves.append(cum)
        fig.add_trace(go.Scatter(
            x=cum.index,
            y=cum,
            mode="lines",
            line=dict(width=1, color=uniform_color),
            hoverinfo="skip",
            showlegend=False
        ))
    except Exception as e:
        print(f"⚠️ {tech}-{retail} ({method}, w={window}) failed: {e}")

# --- Median overlay ---
if all_curves:
    df_all = pd.concat(all_curves, axis=1)
    median_curve = df_all.median(axis=1)
    fig.add_trace(go.Scatter(
        x=median_curve.index,
        y=median_curve,
        mode="lines",
        line=dict(color="orange", width=3),
        name="Median"
    ))

fig.update_layout(
    template="plotly_dark",
    title=f"Top {TOP_N} Mean-Reversion Strategies",
    xaxis_title="Date",
    yaxis_title="Cumulative Return (normalized)",
    showlegend=True,
    height=800
)
fig.show()


In [21]:
# Define output file path
from datetime import datetime
import os

# Create a timestamp for versioning
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# Choose output folder and filename
output_dir = "results"
os.makedirs(output_dir, exist_ok=True)

output_file = f"{output_dir}/cross_grid_results_{timestamp}.csv"

# Save the results sorted by Sharpe ratio
cross_df.sort_values(by="Sharpe", ascending=False).to_csv(output_file, index=False)
print(f"{os.path.abspath(output_file)}")

/Users/vinaypillai/Documents/Northeastern/Clubs/NUSA/fa_25/results/cross_grid_results_2025-11-24_20-47-48.csv


In [22]:
# Load full results
df = pl.read_csv("results/cross_grid_results_2025-11-20_09-18-35.csv")

In [23]:
# Step 1: Filter for Sharpe >= 0.80
filtered = df.filter(pl.col("Sharpe") >= 0.80)

# Step 2: For each (Primary, Pair), keep ONLY the best-performing strategy
best_per_pair = (
    filtered
    .sort("Sharpe", descending=True)
    .group_by(["Primary", "Pair"])
    .agg([
        pl.first("Method").alias("BestMethod"),
        pl.first("Window").alias("BestWindow"),
        pl.first("Sharpe").alias("BestSharpe"),
        pl.first("AnnRet").alias("AnnRet"),
        pl.first("AnnVol").alias("AnnVol"),
        pl.first("Sortino").alias("Sortino"),
        pl.first("MaxDD").alias("MaxDD"),
        pl.first("TotalRet").alias("TotalRet")
    ])
    .sort("BestSharpe", descending=True)
)

print(best_per_pair)

# Save
best_per_pair.write_csv("results/top_pairs_sharpe80_best_strategy.csv")

shape: (25, 10)
┌─────────┬──────┬────────────┬────────────┬───┬──────────┬──────────┬───────────┬──────────┐
│ Primary ┆ Pair ┆ BestMethod ┆ BestWindow ┆ … ┆ AnnVol   ┆ Sortino  ┆ MaxDD     ┆ TotalRet │
│ ---     ┆ ---  ┆ ---        ┆ ---        ┆   ┆ ---      ┆ ---      ┆ ---       ┆ ---      │
│ str     ┆ str  ┆ str        ┆ i64        ┆   ┆ f64      ┆ f64      ┆ f64       ┆ f64      │
╞═════════╪══════╪════════════╪════════════╪═══╪══════════╪══════════╪═══════════╪══════════╡
│ SQ      ┆ LULU ┆ ewma       ┆ 5          ┆ … ┆ 0.115475 ┆ 3.512161 ┆ -0.072824 ┆ 0.270878 │
│ SNOW    ┆ BBY  ┆ standard   ┆ 20         ┆ … ┆ 0.252365 ┆ 3.500014 ┆ -0.193313 ┆ 1.074616 │
│ SQ      ┆ BBY  ┆ standard   ┆ 5          ┆ … ┆ 0.19483  ┆ 2.593498 ┆ -0.116924 ┆ 0.383093 │
│ SQ      ┆ WMT  ┆ standard   ┆ 5          ┆ … ┆ 0.191954 ┆ 2.054753 ┆ -0.144072 ┆ 0.309047 │
│ SNOW    ┆ CVS  ┆ standard   ┆ 20         ┆ … ┆ 0.253828 ┆ 2.27967  ┆ -0.185909 ┆ 0.747686 │
│ …       ┆ …    ┆ …          ┆ …          ┆

In [24]:
# Load the best-per-pair strategies you saved
best_per_pair = pl.read_csv("results/top_pairs_sharpe80_best_strategy.csv")

# Compute Sharpe-squared weights
weighted = (
    best_per_pair
    .with_columns([
        (pl.col("BestSharpe") ** 2).alias("SharpeSq")
    ])
    .with_columns([
        (pl.col("SharpeSq") / pl.col("SharpeSq").sum()).alias("Weight")
    ])
    .sort("Weight", descending=True)
)

print(weighted)

# save the weighting table
weighted.write_csv(f"results/top_pairs_sharpe80_best_strategy_weights_{timestamp}.csv")

shape: (25, 12)
┌─────────┬──────┬────────────┬────────────┬───┬───────────┬──────────┬──────────┬──────────┐
│ Primary ┆ Pair ┆ BestMethod ┆ BestWindow ┆ … ┆ MaxDD     ┆ TotalRet ┆ SharpeSq ┆ Weight   │
│ ---     ┆ ---  ┆ ---        ┆ ---        ┆   ┆ ---       ┆ ---      ┆ ---      ┆ ---      │
│ str     ┆ str  ┆ str        ┆ i64        ┆   ┆ f64       ┆ f64      ┆ f64      ┆ f64      │
╞═════════╪══════╪════════════╪════════════╪═══╪═══════════╪══════════╪══════════╪══════════╡
│ SQ      ┆ LULU ┆ ewma       ┆ 5          ┆ … ┆ -0.072824 ┆ 0.270878 ┆ 1.988784 ┆ 0.080111 │
│ SNOW    ┆ BBY  ┆ standard   ┆ 20         ┆ … ┆ -0.193313 ┆ 1.074616 ┆ 1.878813 ┆ 0.075681 │
│ SQ      ┆ BBY  ┆ standard   ┆ 5          ┆ … ┆ -0.116924 ┆ 0.383093 ┆ 1.620072 ┆ 0.065259 │
│ SQ      ┆ WMT  ┆ standard   ┆ 5          ┆ … ┆ -0.144072 ┆ 0.309047 ┆ 1.087795 ┆ 0.043818 │
│ SNOW    ┆ CVS  ┆ standard   ┆ 20         ┆ … ┆ -0.185909 ┆ 0.747686 ┆ 1.055893 ┆ 0.042533 │
│ …       ┆ …    ┆ …          ┆ …          ┆

In [27]:
# Load weights (best-per-pair Sharpe^2-weighted)
weights = pd.read_csv("results/top_pairs_sharpe80_best_strategy_weights.csv")

# Convert date range
start = START
end   = END

# Build Weighted Index Daily Returns
index_returns_list = []

for _, row in weights.iterrows():
    tech   = row["Primary"]
    retail = row["Pair"]
    method = row["BestMethod"]
    window = int(row["BestWindow"])
    w      = row["Weight"]

    # Replay the strategy
    data, rets = load_prices_cached(tech, retail)
    pnl, _, _  = mean_reversion_strategy(
        data, rets, tech, retail, window, Z_ENTRY, Z_EXIT, method, COSTS
    )

    # Weight its daily pnl
    weighted_pnl = pnl * w
    index_returns_list.append(weighted_pnl)

# Combine all weighted daily returns → Index return series
index_returns = pd.concat(index_returns_list, axis=1).sum(axis=1)
index_returns.name = "IndexReturn"

# Build cumulative NAV
index_nav = (1 + index_returns).cumprod()
index_nav.name = "IndexNAV"

# Load SPY (S&P 500 benchmark)

# Fetch SPY as a single series
spy_prices = fetch_polygon_agg("SPY", pd.Timestamp(START), pd.Timestamp(END))

# Compute returns
spy_returns = spy_prices["SPY"].pct_change().dropna()
spy_returns.name = "SPY_Return"

# Compute NAV
spy_nav = (1 + spy_returns).cumprod()
spy_nav.name = "SPY_NAV"


# Visualization
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=index_nav.index,
    y=index_nav,
    mode='lines',
    name='Strategy Index',
    line=dict(color='cyan')
))

fig.add_trace(go.Scatter(
    x=spy_nav.index,
    y=spy_nav,
    mode='lines',
    name='SPY',
    line=dict(color='orange')
))

fig.update_layout(
    title="Cumulative Returns: Strategy Index vs S&P 500 (2 Years)",
    xaxis_title="Date",
    yaxis_title="Cumulative Return (NAV)",
    template="plotly_dark",
    height=600
)

fig.show()

# Metrics Table
def compute_metrics(r):
    return {
        "AnnReturn": annualize_mean(r),
        "AnnVol": annualize_vol(r),
        "Sharpe": sharpe_ratio(r),
        "MaxDD": max_dd(r)
    }

metrics_index = compute_metrics(index_returns)
metrics_spy   = compute_metrics(spy_returns)

metrics_df = pd.DataFrame({
    "StrategyIndex": metrics_index,
    "SP500": metrics_spy
})

print("\nPerformance Metrics")
print(metrics_df)



Performance Metrics
           StrategyIndex     SP500
AnnReturn       0.264963  0.208173
AnnVol          0.081121  0.163631
Sharpe          2.649922  0.966645
MaxDD          -0.046988 -0.189989


### Scenario Analysis (Across different Cost Assumptions)

In [28]:
from dataclasses import dataclass

Step 1: Define the Cost Scenarios

In [30]:
@dataclass
class CostScenario:
    name: str
    commission_bps: float
    slippage_bps: float
    borrow_bps_per_day: float

scenarios = [
    CostScenario(
        name="Optimistic",
        commission_bps=0.0,
        slippage_bps=1.0,
        borrow_bps_per_day=0.0
    ),
    CostScenario(
        name="Realistic",
        commission_bps=1.0,
        slippage_bps=3.0,
        borrow_bps_per_day=5.0
    ),
    CostScenario(
        name="Hedge Fund Standards",
        commission_bps=2.0,
        slippage_bps=4.0,
        borrow_bps_per_day=10.0
    ),
    CostScenario(
        name="Conservative",
        commission_bps=2.5,
        slippage_bps=2.5,
        borrow_bps_per_day=15.0
    ),
    CostScenario(
        name="Worst-Case",
        commission_bps=3.0,
        slippage_bps=5.0,
        borrow_bps_per_day=20.0
    ),
]


Step 2: Apply the cost function

In [31]:
def apply_costs_to_returns(pnl_returns, positions, cost_scenario):
    """
    Apply trading and borrow costs to daily returns
    
    Parameters:
    - pnl_returns: Series of daily P&L returns (from mean_reversion_strategy)
    - positions: DataFrame of positions for each pair [Primary, Pair columns]
    - cost_scenario: CostScenario object
    
    Returns:
    - Series of net returns after costs
    """
    
    # Trading costs (commission + slippage on turnover)
    turnover = positions.diff().abs().sum(axis=1)
    trade_cost = -(cost_scenario.commission_bps + cost_scenario.slippage_bps) / 1e4 * turnover
    
    # Borrow costs (daily, on short notional)
    borrow_cost = -(cost_scenario.borrow_bps_per_day / 1e4) * positions.clip(upper=0).abs().sum(axis=1)
    
    # Combined cost
    total_cost = trade_cost + borrow_cost
    
    # Apply to returns
    net_returns = pnl_returns + total_cost
    
    return net_returns

Step 3: Generate Returns of the index under each scenario

In [38]:
def run_scenario_analysis(weights, start, end, scenarios):
    """
    Regenerate the index P&L under different cost scenarios
    
    Parameters:
    - weights: DataFrame from top_pairs_sharpe80_best_strategy_weights.csv
    - start, end: date range
    - scenarios: list of CostScenario objects
    
    Returns:
    - Dictionary mapping scenario name to net returns series
    """
    
    scenario_returns = {}
    
    for scenario in scenarios:
        print(f"Running: {scenario.name}")
        print(f"  Commission: {scenario.commission_bps} bps")
        print(f"  Slippage: {scenario.slippage_bps} bps")
        print(f"  Borrow: {scenario.borrow_bps_per_day} bps/day")
        
        index_returns_list = []
        
        for i, row in weights.iterrows():
            tech = row["Primary"]
            retail = row["Pair"]
            method = row["BestMethod"]
            window = int(row["BestWindow"])
            w = row["Weight"]
            
            try:
                # Replay the strategy
                data, rets = load_prices_cached(tech, retail)
                pnl, _, positions = mean_reversion_strategy(
                    data, rets, tech, retail, window, Z_ENTRY, Z_EXIT, method, COSTS
                )
                
                # Apply costs from this scenario
                pnl_net = apply_costs_to_returns(pnl, positions, scenario)
                
                # Weight by Sharpe-squared
                weighted_pnl = pnl_net * w
                index_returns_list.append(weighted_pnl)
                
            except Exception as e:
                print(f"{tech}-{retail} failed: {e}")
                continue
        
        # Combine all weighted daily returns
        index_returns = pd.concat(index_returns_list, axis=1).sum(axis=1)
        scenario_returns[scenario.name] = index_returns
        
        print(f"{scenario.name} complete")
    
    return scenario_returns


Step 4: Compute Metrics for each scenario

In [33]:
def compute_scenario_metrics(scenario_returns, spy_returns):
    """
    Compute performance metrics for each scenario
    
    Returns:
    - DataFrame with metrics: AnnRet, AnnVol, Sharpe, MaxDD
    """
    
    metrics_list = []
    
    for scenario_name, index_ret in scenario_returns.items():
        nav = (1 + index_ret).cumprod()
        
        metrics = {
            "Scenario": scenario_name,
            "AnnReturn": annualize_mean(index_ret),
            "AnnVol": annualize_vol(index_ret),
            "Sharpe": sharpe_ratio(index_ret),
            "Sortino": sortino_ratio(index_ret),
            "MaxDD": max_dd(index_ret),
            "TotalReturn": (1 + index_ret).prod() - 1,
        }
        metrics_list.append(metrics)
    
    # Add SPY benchmark
    metrics_list.append({
        "Scenario": "SPY Benchmark",
        "AnnReturn": annualize_mean(spy_returns),
        "AnnVol": annualize_vol(spy_returns),
        "Sharpe": sharpe_ratio(spy_returns),
        "Sortino": sortino_ratio(spy_returns),
        "MaxDD": max_dd(spy_returns),
        "TotalReturn": (1 + spy_returns).prod() - 1,
    })
    
    return pd.DataFrame(metrics_list)

Step 5: Plot the scenario analysis

In [34]:
def plot_scenario_analysis(scenario_returns, spy_returns, start, end):
    """
    Plot cumulative returns for all scenarios
    """
    
    fig = go.Figure()
    
    # Plot each scenario
    colors = [
        "cyan",      # Optimistic
        "orange",    # Realistic Retail
        "purple",    # HF
        "green",     # Conservative
        "red",       # Worst-case
    ]
    
    for (scenario_name, returns), color in zip(scenario_returns.items(), colors):
        nav = (1 + returns).cumprod()
        fig.add_trace(go.Scatter(
            x=nav.index,
            y=nav,
            mode='lines',
            name=scenario_name,
            line=dict(width=2, color=color)
        ))
    
    # Add SPY benchmark
    spy_nav = (1 + spy_returns).cumprod()
    fig.add_trace(go.Scatter(
        x=spy_nav.index,
        y=spy_nav,
        mode='lines',
        name='SPY (Benchmark)',
        line=dict(width=2, color='gray', dash='dash')
    ))
    
    fig.update_layout(
        title="Strategy Performance Across Cost Scenarios",
        xaxis_title="Date",
        yaxis_title="Cumulative Return (NAV)",
        template="plotly_dark",
        height=700,
        hovermode='x unified'
    )
    
    fig.show()

Step 6: Generate the Summary Table

In [35]:
def print_scenario_summary(metrics_df):
    """
    Print formatted scenario comparison
    """
    print("SCENARIO ANALYSIS SUMMARY")
    
    display_df = metrics_df.copy()
    
    print(display_df.to_string(
        formatters={
            "AnnReturn": "{:.2%}".format,
            "AnnVol": "{:.2%}".format,
            "Sharpe": "{:.2f}".format,
            "Sortino": "{:.2f}".format,
            "MaxDD": "{:.2%}".format,
            "TotalReturn": "{:.2%}".format,
        },
        index=False
    ))

Step 7: Calculate Impact vs Optimistic 

In [36]:
def calculate_drag(metrics_df):
    """
    Calculate the cost drag relative to Optimistic scenario
    """
    
    optimistic_idx = metrics_df[metrics_df["Scenario"] == "Optimistic"].index[0]
    optimistic_sharpe = metrics_df.loc[optimistic_idx, "Sharpe"]
    optimistic_ret = metrics_df.loc[optimistic_idx, "AnnReturn"]
    
    print("\n")
    print("COST IMPACT ANALYSIS")
    
    for _, row in metrics_df.iterrows():
        if row["Scenario"] == "Optimistic":
            continue
        
        sharpe_drag = optimistic_sharpe - row["Sharpe"]
        return_drag = optimistic_ret - row["AnnReturn"]
        
        print(f"\n{row['Scenario']}:")
        print(f"  Sharpe drag: {sharpe_drag:.2f} ({sharpe_drag/optimistic_sharpe*100:.1f}% reduction)")
        print(f"  Return drag: {return_drag:.2%} ({return_drag/optimistic_ret*100:.1f}% reduction)")
        print(f"  Resulting Sharpe: {row['Sharpe']:.2f}")
        print(f"  Resulting Ann Return: {row['AnnReturn']:.2%}")

Step 8: Execute Full Scenario Analysis

In [39]:
if __name__ == "__main__":
    
    print("\nRunning Scenario Analysis")
    
    # Load weights
    weights = pd.read_csv("results/top_pairs_sharpe80_best_strategy_weights.csv")
    
    # Run all scenarios
    scenario_returns = run_scenario_analysis(weights, START, END, scenarios)
    
    # Compute metrics
    metrics_df = compute_scenario_metrics(scenario_returns, spy_returns)
    
    # Display results
    print_scenario_summary(metrics_df)
    calculate_drag(metrics_df)
    
    # Plot
    plot_scenario_analysis(scenario_returns, spy_returns, START, END)
    
    # Save results
    metrics_df.to_csv("results/scenario_analysis_results.csv", index=False)
    print(f"\nScenario analysis saved to: results/scenario_analysis_results.csv")


Running Scenario Analysis
Running: Optimistic
  Commission: 0.0 bps
  Slippage: 1.0 bps
  Borrow: 0.0 bps/day
Optimistic complete
Running: Realistic
  Commission: 1.0 bps
  Slippage: 3.0 bps
  Borrow: 5.0 bps/day
Realistic complete
Running: Hedge Fund Standards
  Commission: 2.0 bps
  Slippage: 4.0 bps
  Borrow: 10.0 bps/day
Hedge Fund Standards complete
Running: Conservative
  Commission: 2.5 bps
  Slippage: 2.5 bps
  Borrow: 15.0 bps/day
Conservative complete
Running: Worst-Case
  Commission: 3.0 bps
  Slippage: 5.0 bps
  Borrow: 20.0 bps/day
Worst-Case complete
SCENARIO ANALYSIS SUMMARY
            Scenario AnnReturn AnnVol Sharpe Sortino   MaxDD TotalReturn
          Optimistic    26.12%  8.11%   2.60    7.20  -4.72%      67.46%
           Realistic    21.99%  8.10%   2.10    5.92  -5.25%      54.21%
Hedge Fund Standards    18.25%  8.10%   1.64    4.81  -5.76%      43.09%
        Conservative    15.63%  8.11%   1.31    4.06  -6.21%      35.81%
          Worst-Case    11.51%  8.10%


Scenario analysis saved to: results/scenario_analysis_results.csv
